In [ ]:
!pip install autogluon pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [ ]:
import pandas as pd
import numpy as np

from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('/content/df_all_clean (1).csv')

print(df.shape)
df.head()

(7333, 25)


,discode,Sex,agey,agem,aged,nation,occupat,addercode,metropol,treatmentloc,...,datefound,datedeath,germ,complications,district,province,death,days_to_death,days_to_define,days_to_report
0,66.0,หญิง,17,5,19,ไทย,ไม่ทราบอาชีพ,26020302,อบต,ร.พ.ชุมชน,...,2018-01-01,NaN,NaN,NaN,ปากพลี,นครนายก,0.0,NaN,0.0,0.0
1,66.0,ชาย,52,8,1,ไทย,รับจ้าง-กรรมกร,14081305,ในเขตเทศบาล,ร.พ.ชุมชน,...,2018-01-01,NaN,NaN,NaN,ผักไห่,พระนครศรีอยุธยา,0.0,NaN,0.0,0.0
2,66.0,ชาย,55,0,0,ไทย,รับจ้าง-กรรมกร,14110303,ในเขตเทศบาล,คลินิก-ร.พ.เอกชน,...,2018-01-01,NaN,9.0,0.0,วังน้อย,พระนครศรีอยุธยา,0.0,NaN,0.0,0.0
3,66.0,ชาย,39,0,0,ไทย,รับจ้าง-กรรมกร,14100605,อบต,ร.พ.ชุมชน,...,2018-01-01,NaN,0.0,0.0,ลาดบัวหลวง,พระนครศรีอยุธยา,0.0,NaN,0.0,0.0
4,66.0,ชาย,51,3,24,ไทย,รับจ้าง-กรรมกร,14051102,ในเขตเทศบาล,ร.พ.ชุมชน,...,2018-01-01,NaN,NaN,NaN,บางบาล,พระนครศรีอยุธยา,0.0,NaN,0.0,0.0


In [ ]:
df.columns

Index(['discode', 'Sex', 'agey', 'agem', 'aged', 'nation', 'occupat',
       'addercode', 'metropol', 'treatmentloc', 'patienttype', 'results',
       'treatmentloccod', 'datesick', 'datedefine', 'datefound', 'datedeath',
       'germ', 'complications', 'district', 'province', 'death',
       'days_to_death', 'days_to_define', 'days_to_report'],
      dtype='object')

In [ ]:
# change to datetime

df["datesick"] = pd.to_datetime(df["datesick"], errors="coerce")
df["datefound"] = pd.to_datetime(df["datefound"], errors="coerce")

In [ ]:
# วันในสัปดาห์ที่ report [ 0 : Monday , 6 :	Sunday ]

df["report_weekday"] = df["datefound"].dt.weekday

In [ ]:
# เดือนที่มีการ report

df["report_month"] = df["datefound"].dt.month

In [ ]:
# จำนวนเคสในวันเดียวกัน

df["cases_same_day"] = (
    df.groupby("datesick")["datesick"]
    .transform("count")
)

In [ ]:
# จำนวนเคสในอำเภอในวันเดียวกัน

df["district_cases_day"] = (
    df.groupby(["district","datesick"])["datesick"]
    .transform("count")
)

In [ ]:
# จำนวนเคสทั้งหมดที่รักษาในสถานพยาบาลนั้น

df["hospital_load"] = (
    df.groupby("treatmentloc")["treatmentloc"]
    .transform("count")
)

In [ ]:
# วันในสัปดาห์ที่ผู้ป่วยเริ่มป่วย

df["sick_weekday"] = df["datesick"].dt.weekday

In [ ]:
# เริ่มป่วย weekend มั้ย [1 : weekend , 0 : weekday]

df["is_weekend_sick"] = df["sick_weekday"].isin([5,6]).astype(int)

In [ ]:
# จำนวนเคสทั้งหมดในจังหวัด

province_cases = df.groupby("province").size()

df["province_case_count"] = df["province"].map(province_cases)

In [ ]:
# จำนวนเคสทั้งหมดในอำเภอ

district_cases = df.groupby("district").size()

df["district_case_count"] = df["district"].map(district_cases)

In [ ]:
# เลือก Feature

features = [
"agey",
"Sex",
"province",
"district",
"occupat",
"discode",
"treatmentloc",
"cases_same_day",
"district_cases_day",
"hospital_load",
"province_case_count",
"district_case_count",
"report_weekday",
"report_month",
"is_weekend_sick",
"days_to_report"
]

In [ ]:
# เตรียม Dataset

df_model = df[features]

In [ ]:
# ลบ Missing Target

df_model = df_model.dropna(subset=["days_to_report"])

In [ ]:
# Split Dataset

train_data, test_data = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42
)

In [ ]:
# train model

predictor = TabularPredictor(
    label="days_to_report",
    problem_type="regression",
    eval_metric="mae"
).fit(
    train_data,
    presets="high_quality"
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260315_045809"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.91 GB / 12.67 GB (86.1%)
Disk Space Avail:   74.75 GB / 107.72 GB (69.4%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if m

(_ray_fit pid=8276) [1000]	valid_set's l1: 2.23779


(_dystack pid=3067) 	-2.0432	 = Validation score   (-mean_absolute_error)
(_dystack pid=3067) 	36.58s	 = Training   runtime
(_dystack pid=3067) 	0.44s	 = Validation runtime
(_dystack pid=3067) Fitting model: LightGBM_BAG_L2 ... Training model for up to 231.74s of the 231.64s of remaining time.
(_dystack pid=3067) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.23%)
(_ray_fit pid=7585) 	Ran out of time, stopping training early. (Stopping on epoch 40)
(_dystack pid=3067) 	-2.1463	 = Validation score   (-mean_absolute_error)
(_dystack pid=3067) 	36.52s	 = Training   runtime
(_dystack pid=3067) 	0.25s	 = Validation runtime
(_dystack pid=3067) Fitting model: RandomForestMSE_BAG_L2 ... Training model for up to 190.23s of the 190.13s of remaining time.
(_dystack pid=3067) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0, mem=0.0/9.2 GB
(_dystack pid=3067) 	-1.9033	 = Validation score

In [ ]:
# model Leaderboard

predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetTorch_BAG_L2_FULL,-1.882306,NaN,mean_absolute_error,0.833651,NaN,183.471585,0.046261,NaN,30.136286,2,True,47
1,WeightedEnsemble_L3_FULL,-1.887567,NaN,mean_absolute_error,1.014780,NaN,196.647998,0.004291,NaN,0.392760,3,True,50
2,WeightedEnsemble_L2_FULL,-1.902673,NaN,mean_absolute_error,0.327903,NaN,111.782835,0.003682,NaN,0.335522,2,True,39
3,CatBoost_BAG_L2_FULL,-1.903830,NaN,mean_absolute_error,0.798646,NaN,160.594937,0.011256,NaN,7.259638,2,True,43
4,XGBoost_BAG_L1_FULL,-1.914598,NaN,mean_absolute_error,0.021953,NaN,0.322159,0.021953,NaN,0.322159,1,True,32
5,RandomForestMSE_BAG_L1_FULL,-1.923632,NaN,mean_absolute_error,0.191517,0.417942,4.386607,0.191517,0.417942,4.386607,1,True,28
6,RandomForestMSE_BAG_L1,-1.923632,-1.971788,mean_absolute_error,0.275878,0.417942,4.386607,0.275878,0.417942,4.386607,1,True,3
7,XGBoost_BAG_L2_FULL,-1.958460,NaN,mean_absolute_error,0.806741,NaN,153.652994,0.019351,NaN,0.317695,2,True,46
8,ExtraTreesMSE_BAG_L1_FULL,-1.964682,NaN,mean_absolute_error,0.183116,0.520118,3.447837,0.183116,0.520118,3.447837,1,True,30
9,ExtraTreesMSE_BAG_L1,-1.964682,-1.876189,mean_absolute_error,0.200977,0.520118,3.447837,0.200977,0.520118,3.447837,1,True,5


In [ ]:
# test Prediction

pred = predictor.predict(test_data)

In [ ]:
# check Feature Importance

predictor.feature_importance(test_data)

These features in provided data are not utilized by the predictor and will be ignored: ['discode']
Computing feature importance via permutation shuffling for 14 features using 1467 rows with 5 shuffle sets...
	75.23s	= Expected runtime (15.05s per shuffle set)
	35.09s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
report_month,1.526370,0.026871,1.152088e-08,5,1.581698,1.471043
Sex,0.988949,0.065114,2.242233e-06,5,1.123020,0.854879
district,0.813953,0.094947,2.182087e-05,5,1.009451,0.618456
cases_same_day,0.722902,0.048038,2.326261e-06,5,0.821813,0.623991
province,0.584762,0.042420,3.300007e-06,5,0.672105,0.497418
report_weekday,0.444323,0.104136,3.370088e-04,5,0.658741,0.229904
treatmentloc,0.227016,0.048181,2.295177e-04,5,0.326222,0.127811
district_case_count,0.139649,0.028220,1.896535e-04,5,0.197754,0.081544
hospital_load,0.129554,0.022701,1.086449e-04,5,0.176296,0.082812
province_case_count,0.128841,0.019088,5.615254e-05,5,0.168142,0.089539


In [26]:
# Save training folder from Colab to Google Drive

from google.colab import drive
import os

# 1) Mount Google Drive
drive.mount('/content/drive')

# 2) กำหนด path ของ folder ที่ต้องการ save
source_folder = "/content/AutogluonModels"   # เปลี่ยนเป็น folder ที่คุณต้องการ
zip_name = "trained_model.zip"

# 3) zip folder
!zip -r $zip_name $source_folder

# 4) copy zip file ไป Google Drive
drive_path = "/content/drive/MyDrive/"
!cp $zip_name $drive_path

# 5) แสดงผลลัพธ์
print("Saved to Google Drive:", drive_path + zip_name)

Mounted at /content/drive
  adding: content/AutogluonModels/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/learner.pkl (deflated 67%)
  adding: content/AutogluonModels/ag-20260315_045809/version.txt (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/predictor.pkl (deflated 50%)
  adding: content/AutogluonModels/ag-20260315_045809/models/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/models/NeuralNetFastAI_BAG_L1/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/models/NeuralNetFastAI_BAG_L1/S1F3/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/models/NeuralNetFastAI_BAG_L1/S1F3/model.pkl (deflated 47%)
  adding: content/AutogluonModels/ag-20260315_045809/models/NeuralNetFastAI_BAG_L1/S1F2/ (stored 0%)
  adding: content/AutogluonModels/ag-20260315_045809/models/NeuralNetFastAI_BAG_L1/S1F2/model.pkl (deflated 47%)
  adding: content/Autoglu